## Data Preprocessing & Cleaning
In this step, we have to clean the data because raw text from the internet is messy. By cleaning the text, we standardize our vocabulary so that the deep learning model can focus purely on the semantic meaning of the words rather than getting distracted by typos, formatting bugs, or casing differences.

In [2]:
import os
import zipfile
import pandas as pd
import html  

# 1. loading the raw data
zip_path = os.path.join('..', 'dataset', 'archive.zip')

with zipfile.ZipFile(zip_path, 'r') as zip_file:
    df_train = pd.read_csv(zip_file.open('drugsComTrain_raw.csv'))
    df_test = pd.read_csv(zip_file.open('drugsComTest_raw.csv'))

# 2. Droping rows where essential data is missing
df_train = df_train.dropna(subset=['review', 'rating'])
df_test = df_test.dropna(subset=['review', 'rating'])

# 3. Defining our custom text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # Decoding HTML entities 
    text = html.unescape(text)
    
    # Standardizing to lowercase
    text = text.lower()
    
    # Striping extra whitespace trailing elements
    text = text.strip()
    
    return text

print("Cleaning training reviews...")
df_train['clean_review'] = df_train['review'].apply(clean_text)

print("Cleaning test reviews...")
df_test['clean_review'] = df_test['review'].apply(clean_text)

print("\nSample comparison:")
print("RAW   :", df_train['review'].iloc[0][:100])
print("CLEAN :", df_train['clean_review'].iloc[0][:100])

Cleaning training reviews...
Cleaning test reviews...

Sample comparison:
RAW   : "It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"
CLEAN : "it has no side effect, i take it in combination of bystolic 5 mg and fish oil"


### Data Tokenization & Vocabulary Building
In this step, we transition from human words to numbers that neural network can process.

In [3]:
import setuptools
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# 1. Define hyperparameters from our Milestone 2 EDA
VOCAB_SIZE = 10000  # Max vocabulary cutoff
MAX_LEN = 150       # Our optimized sequence length cutoff

# 2. Create the modern vectorization layer
# This completely replaces the old Tokenizer and pad_sequences modules
vectorize_layer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)

print("Building vocabulary from clean training data...")
# Convert our pandas text column into a native TensorFlow dataset for fast processing
train_text_ds = tf.data.Dataset.from_tensor_slices(df_train['clean_review'].values).batch(128)
vectorize_layer.adapt(train_text_ds)

print("Transforming text into padded integer matrices...")
# Process the reviews into matrices
X_train_padded = vectorize_layer(df_train['clean_review'].values).numpy()
X_test_padded = vectorize_layer(df_test['clean_review'].values).numpy()

print("\n--- Tokenization & Padding Successful! ---")
print(f"Shape of padded training matrix: {X_train_padded.shape}")
print(f"Shape of padded test matrix: {X_test_padded.shape}")

# Verify what the first row looks like now
print("\nFirst review as an integer array (showing first 20 tokens):")
print(X_train_padded[0][:20])

Building vocabulary from clean training data...


2026-06-05 00:43:51.324936: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Transforming text into padded integer matrices...

--- Tokenization & Padding Successful! ---
Shape of padded training matrix: (161297, 150)
Shape of padded test matrix: (53766, 150)

First review as an integer array (showing first 20 tokens):
[   8   37   27   34  193    2   43    8   15  835   12 2919  144  148
    3 3823 1570    0    0    0]


### Saving the Processed Data

In [4]:
import os
import numpy as np

# Defining our destination directory for processed data
processed_dir = os.path.join('..', 'dataset', 'processed')
os.makedirs(processed_dir, exist_ok=True)

# Preparing our target labels (the ratings) as numpy arrays
y_train = df_train['rating'].values
y_test = df_test['rating'].values

# Saving everything into a compressed file format
save_path = os.path.join(processed_dir, 'processed_data.npz')
np.savez_compressed(
    save_path,
    X_train=X_train_padded,
    X_test=X_test_padded,
    y_train=y_train,
    y_test=y_test
)

print(f"Success! Processed datasets saved to: {save_path}")
print(f"Stored Arrays: X_train {X_train_padded.shape}, y_train {y_train.shape}")

Success! Processed datasets saved to: ../dataset/processed/processed_data.npz
Stored Arrays: X_train (161297, 150), y_train (161297,)
